# 15 — Heterogeneous Graph Transformer：在公开 DBLP 上学习异构注意力

上一课的异构 GraphSAGE 已经为每种关系使用独立参数，但同一关系中的邻居仍通过固定 mean 聚合。本课使用 HGT，让模型根据源节点类型、目标节点类型、关系类型和节点内容，为每条邻边自适应计算注意力。实验使用完整公开 DBLP，不再使用 toy 数据。

In [ ]:
from pathlib import Path
import sys
import time
import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from crosscity.data.heterogeneous_node_classification import load_dblp
from crosscity.models.heterogeneous_transformer import (
    HeterogeneousMLPClassifier, HeterogeneousSAGEClassifier, HGTClassifier,
)
from crosscity.training.heterogeneous_node_classification import train_heterogeneous_node_classifier
from crosscity.utils import seed_everything
if torch.cuda.is_available(): device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else: device = torch.device('cpu')
print('device:', device)
if device.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

## 1. DBLP 是什么任务？

DBLP 是计算机科学论文目录。本数据集包含：

- `author`：4,057 位作者，目标节点；
- `paper`：14,328 篇论文；
- `term`：7,723 个关键词；
- `conference`：20 个会议。

边表示作者写论文、论文包含关键词、论文发表于会议，以及相应反向边。任务根据作者自身词袋特征及异构邻居，预测四个研究领域：数据库、数据挖掘、人工智能、信息检索。它是节点分类，不是推荐；这样我们可以先专注理解 HGT 编码器，下一步再把 HGT 表示接回商品排序解码器。

In [ ]:
graph = load_dblp(ROOT / 'data/raw/dblp')
print(graph)
rows = []
for node_type in graph.node_types:
    store = graph[node_type]
    rows.append({
        'node_type': node_type, 'nodes': store.num_nodes,
        'feature_dim': store.x.size(1) if 'x' in store else None,
    })
display(pd.DataFrame(rows))
pd.DataFrame([{
    'edge_type': str(edge_type), 'edges': graph[edge_type].num_edges
} for edge_type in graph.edge_types])

## 2. 不同节点为什么必须先投影？

作者、论文和关键词的原始特征维度不同，会议甚至没有输入特征。消息相加前必须进入统一的 $d$ 维隐藏空间：

$$h_v^{(0)}=ReLU(W_{\tau(v)}x_v),$$

其中 $\tau(v)$ 是节点类型，每种类型拥有独立投影 $W_{\tau}$。没有特征的会议节点使用可学习 ID embedding。类型专属投影不是注意力，它只是把不同来源的信息转换到可以运算的共同空间。

In [ ]:
input_dims = {
    node_type: graph[node_type].x.size(1) if 'x' in graph[node_type] else None
    for node_type in graph.node_types
}
num_nodes = {node_type: graph[node_type].num_nodes for node_type in graph.node_types}
print('input dimensions:', input_dims)
print('conference has no x, so it will use an ID embedding')

## 3. 从 GAT 到 HGT：注意力到底在学什么？

普通 GAT 在同一种节点和边上，为邻居 $s$ 到目标 $t$ 计算分数，再 softmax：

$$\alpha_{s,t}=softmax_{s\in\mathcal N(t)}(score(h_s,h_t))$$

所以注意力不是直接存一张固定边权表，而是由当前节点表示动态算出。不同作者连接同一论文时，权重可以不同；训练后节点表示改变，权重也会改变。

异构图还多了类型和关系。HGT 把一条边写成元关系：

$$m=(\tau(s),\phi(e),\tau(t))$$

例如 `(paper, to, author)` 与 `(term, to, paper)` 即使关系字符串都叫 `to`，源/目标类型不同，仍是不同元关系。

## 4. HGT 注意力的数学分解

对第 $i$ 个注意力头，目标节点产生 Query，源节点产生 Key：

$$q_t^i=Q_{\tau(t)}^i h_t,\qquad k_s^i=K_{\tau(s)}^i h_s$$

注意 $Q$ 和 $K$ 取决于节点类型：论文怎样提出问题、作者怎样作为邻居提供匹配信息，可以使用不同投影。关系再用矩阵 $W_{ATT}^{\phi(e)}$ 修改匹配方式：

$$a_{s,e,t}^i=\frac{(k_s^i)^T W_{ATT}^{\phi(e)}q_t^i}{\sqrt{d_h}}$$

同一目标的所有入边做 softmax：

$$\alpha_{s,e,t}^i=\frac{\exp(a_{s,e,t}^i)}{\sum_{s'\in\mathcal N(t)}\exp(a_{s',e',t}^i)}$$

所以 $\alpha$ 才是每条邻边最终自适应得到的权重；$W_{ATT}$ 是训练得到的关系级规则。

## 5. 注意力权重和消息内容不是一回事

HGT 另外生成 Value/Message：

$$v_s^i=V_{\tau(s)}^i h_s$$
$$m_{s,e,t}^i=W_{MSG}^{\phi(e)}v_s^i$$

$W_{ATT}$ 决定“应该听多少”，$W_{MSG}$ 决定“传来的内容如何解释”。聚合为：

$$\tilde h_t=\mathop{Concat}_i\left(\sum_{s\in\mathcal N(t)}\alpha_{s,e,t}^i m_{s,e,t}^i\right)$$

最后通过目标类型专属输出变换并加残差：

$$h_t'=LayerNorm(h_t+A_{\tau(t)}\tilde h_t)$$

因此 HGT 同时感知源类型、目标类型、关系类型和具体节点内容。

## 6. 多头注意力是什么？

若隐藏维度 $d=64$、`heads=4`，每个头在 $d_h=16$ 维子空间计算注意力。不同头可以形成不同关注模式，例如一个头偏向论文关键词，另一个头偏向会议信息。各头结果拼接回 64 维。

多头并不保证每个头都会自动对应人类可解释概念，也不是把参数简单复制四份投票；它让模型拥有多个并行的匹配子空间。隐藏维度必须能整除 head 数。

## 7. HGT、R-GCN 和异构 GraphSAGE 的区别

| 模型 | 是否区分关系 | 同一关系内邻居权重 | 是否区分节点类型投影 |
|---|---:|---|---:|
| R-GCN | 是 | 固定归一化 | 通常统一实体空间 |
| Hetero GraphSAGE | 是 | mean | 是 |
| HGT | 是 | 内容相关 softmax 注意力 | 是 |

HGT 的优势是表达力，代价是计算、显存和过拟合风险更高。因此 MLP 和关系分组 GraphSAGE 仍然是必要基线。

## 8. Accuracy 与 Macro-F1

Accuracy 是预测正确作者数除以作者总数：

$$Accuracy=\frac{correct}{N}$$

对类别 $c$，$Precision_c=TP_c/(TP_c+FP_c)$，$Recall_c=TP_c/(TP_c+FN_c)$，再计算：

$$F1_c=\frac{2Precision_c Recall_c}{Precision_c+Recall_c}$$
$$MacroF1=\frac{1}{C}\sum_cF1_c$$

Macro-F1 对四个研究领域等权，不会让样本多的领域主导结果。本课按 validation Macro-F1 早停，只在最后读取 test。

In [ ]:
RUN_FULL = False
seeds = [42, 43, 44] if RUN_FULL else [42]
max_epochs = 150 if RUN_FULL else 100
patience = 30
hidden_channels = 64
num_classes = int(graph['author'].y.max()) + 1

def model_builders():
    return {
        'AuthorMLP': lambda: HeterogeneousMLPClassifier(
            input_dims['author'], hidden_channels, num_classes),
        'HeteroSAGE': lambda: HeterogeneousSAGEClassifier(
            input_dims, num_nodes, graph.metadata(), hidden_channels,
            num_classes, num_layers=2),
        'HGT': lambda: HGTClassifier(
            input_dims, num_nodes, graph.metadata(), hidden_channels,
            num_classes, num_layers=2, heads=4),
    }
print('runs:', len(seeds) * 3, '| full public graph | device:', device)

In [ ]:
rows, histories = [], {}
for seed in seeds:
    for name, build in model_builders().items():
        seed_everything(seed)
        model = build()
        if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        result = train_heterogeneous_node_classifier(
            model, graph, device=device, max_epochs=max_epochs, patience=patience
        )
        elapsed = time.perf_counter() - started
        peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if device.type == 'cuda' else float('nan')
        rows.append({
            'seed': seed, 'model': name, 'best_epoch': result.best_epoch,
            'validation_accuracy': result.validation_accuracy,
            'validation_macro_f1': result.validation_macro_f1,
            'test_accuracy': result.test_accuracy, 'test_macro_f1': result.test_macro_f1,
            'seconds': elapsed, 'peak_cuda_mb': peak_mb,
        })
        histories[(seed, name)] = result.history
results = pd.DataFrame(rows)
display(results.sort_values('validation_macro_f1', ascending=False))
results.groupby('model')[['validation_macro_f1', 'test_macro_f1', 'seconds', 'peak_cuda_mb']].agg(['mean', 'std'])

In [ ]:
for (seed, name), history in histories.items():
    if seed == seeds[0]:
        frame = pd.DataFrame(history)
        plt.plot(frame.epoch, frame.validation_macro_f1, label=name)
plt.xlabel('epoch'); plt.ylabel('validation macro-F1')
plt.title('DBLP: fixed relation aggregation vs heterogeneous attention')
plt.legend(); plt.show()

## 9. 如何解释实验

- `HeteroSAGE > AuthorMLP`：论文、关键词和会议关系提供了作者自身特征之外的信息。
- `HGT > HeteroSAGE`：内容相关注意力比固定 mean/sum 更适合当前关系。
- `HGT ≈ HeteroSAGE`：关系分组已经足够，额外注意力复杂度未产生稳定收益。
- HGT 单 seed 更好但三 seed 波动重叠：不能宣称注意力稳定胜出。
- 同时比较耗时和显存：准确率提升很小却资源大增时，工程上未必值得。

本 notebook 是完整图 full-batch 训练。DBLP 规模适中，可以放进常见 CUDA 显存；更大的 OGB-MAG 需要异构邻居采样，不能仅靠增加 GPU。

## 10. 时间戳与时空图是什么关系？

时间戳首先规定信息可见性：预测 $t$ 时刻购买，只能使用时间 $<t$ 的点击、加购和购买。它还描述兴趣漂移——昨天搜索相机和半年前买过书，重要性不同。Temporal GNN 会进一步让消息依赖时间差 $\Delta t$，或按事件顺序更新节点记忆。

交通时空图则通常有固定空间节点和连续时间信号。两者共同点是图结构与时间依赖同时存在；区别是电商多为不规则离散事件，交通多为规则采样序列。后续课程会先讲 Temporal GNN，再把它和早期 STGCN 对照。

## 本课结论

1. HGT 的边权由节点内容、节点类型和关系类型共同动态计算。
2. $W_{ATT}$ 决定如何匹配，$W_{MSG}$ 决定如何解释消息，$\alpha$ 才是具体边权。
3. 多头注意力提供多个匹配子空间，不保证天然可解释。
4. HGT 表达力更强，也更耗显存、更容易过拟合。
5. 公开 DBLP 实验必须保留 MLP 和固定聚合基线，并报告多 seed、耗时和显存。
6. 下一课将进入带时间戳的动态图：从静态 `edge_index` 转向按时间到达的事件流。